In [1]:
import pandas as pd
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests
import numpy as np
import semopy
import json

In [2]:
colunas_interesse = ["Modelo", "RMSEA", "AIC", "BIC"]
df_modelos_controle_prep = pd.read_csv(r"..\resultados\resultados_empiricos\modelos_controle_prep.csv", index_col="Modelo", usecols=colunas_interesse)
df_modelos_autismo_prep = pd.read_csv(r"..\resultados\resultados_empiricos\modelos_autismo_prep.csv", index_col="Modelo", usecols=colunas_interesse)

df_controle_prep = pd.read_csv(r"..\dados\raios_espectrais\raios_espectrais_controle_prep.csv")
df_autismo_prep = pd.read_csv(r"..\dados\raios_espectrais\raios_espectrais_autismo_prep.csv")

with open(r"..\utils\fm_modelos_5_conexo.json", "r") as file:
    formula_modelos_conexos = json.load(file)

### 1) Filtragem de modelos com base em RMSEA < 0.08

In [3]:
df_controle_filtro1 = df_modelos_controle_prep[df_modelos_controle_prep["RMSEA"] < 0.08]
df_autismo_filtro1 = df_modelos_autismo_prep[df_modelos_autismo_prep["RMSEA"] < 0.08]

print(df_controle_filtro1.shape)
print(df_autismo_filtro1.shape)

(375, 3)
(359, 3)


In [4]:
df_controle_filtro1.sort_values("BIC").head(10)

,RMSEA,AIC,BIC
Modelo,,,
M08682_Edges4,0.071795,17.915579,56.354475
M08812_Edges4,0.071795,17.915579,56.354475
M03528_Edges4,0.000000,17.999997,56.438893
M00934_Edges4,0.000000,18.000000,56.438895
M00001_Edges4,0.000000,18.000000,56.438896
M25888_Edges4,0.000000,18.000000,56.438896
M10302_Edges4,0.000000,18.000000,56.438896
M09102_Edges5,0.079957,19.917286,62.627170
M08640_Edges5,0.079957,19.917286,62.627170


In [5]:
df_autismo_filtro1.sort_values("BIC").head(10)

,RMSEA,AIC,BIC
Modelo,,,
M08682_Edges4,0.067727,17.903044,50.775447
M08812_Edges4,0.067727,17.903045,50.775447
M00001_Edges4,0.000000,17.999999,50.872402
M00934_Edges4,0.000000,18.000000,50.872403
M03528_Edges4,0.000000,18.000000,50.872403
M10302_Edges4,0.000000,18.000000,50.872403
M25888_Edges4,0.000000,18.000000,50.872403
M25373_Edges5,0.058305,19.931037,56.455929
M25243_Edges5,0.058305,19.931037,56.455929


### 2) Selecionando modelos que passaram na etapa 1, simultaneamente para ambos os conjuntos

In [6]:
modelos_filtro2 = pd.merge(df_controle_filtro1, 
                           df_autismo_filtro1, 
                           how="inner", 
                           left_index=True, 
                           right_index=True, 
                           suffixes=["_controle", "_autismo"])
modelos_filtro2.shape

(255, 6)

### 3) Verificar os TOP 10 modelos com base em ordenação de AIC/BIC

#### 3.1) Ordenação com base no AIC/BIC produzido pelo Conjunto Controle:

In [7]:
modelos_filtro2.sort_values("BIC_controle").head(10)

,RMSEA_controle,AIC_controle,BIC_controle,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,
M08682_Edges4,0.071795,17.915579,56.354475,0.067727,17.903044,50.775447
M08812_Edges4,0.071795,17.915579,56.354475,0.067727,17.903045,50.775447
M03528_Edges4,0.000000,17.999997,56.438893,0.000000,18.000000,50.872403
M00934_Edges4,0.000000,18.000000,56.438895,0.000000,18.000000,50.872403
M00001_Edges4,0.000000,18.000000,56.438896,0.000000,17.999999,50.872402
M25888_Edges4,0.000000,18.000000,56.438896,0.000000,18.000000,50.872403
M10302_Edges4,0.000000,18.000000,56.438896,0.000000,18.000000,50.872403
M09102_Edges5,0.079957,19.917286,62.627170,0.054826,19.934959,56.459850
M08640_Edges5,0.079957,19.917286,62.627170,0.054826,19.934959,56.459851


Percebe-se que o Top 7 tem uma discrepância em BIC relevante a partir do 8º modelo

In [8]:
modelos_filtro2.sort_values("BIC_controle").head(7)

,RMSEA_controle,AIC_controle,BIC_controle,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,
M08682_Edges4,0.071795,17.915579,56.354475,0.067727,17.903044,50.775447
M08812_Edges4,0.071795,17.915579,56.354475,0.067727,17.903045,50.775447
M03528_Edges4,0.000000,17.999997,56.438893,0.000000,18.000000,50.872403
M00934_Edges4,0.000000,18.000000,56.438895,0.000000,18.000000,50.872403
M00001_Edges4,0.000000,18.000000,56.438896,0.000000,17.999999,50.872402
M25888_Edges4,0.000000,18.000000,56.438896,0.000000,18.000000,50.872403
M10302_Edges4,0.000000,18.000000,56.438896,0.000000,18.000000,50.872403


#### 3.2) Ordenação com base no AIC/BIC produzido pelo Conjunto Autismo:

In [9]:
modelos_filtro2.sort_values("BIC_autismo").head(10)

,RMSEA_controle,AIC_controle,BIC_controle,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,
M08682_Edges4,0.071795,17.915579,56.354475,0.067727,17.903044,50.775447
M08812_Edges4,0.071795,17.915579,56.354475,0.067727,17.903045,50.775447
M00001_Edges4,0.000000,18.000000,56.438896,0.000000,17.999999,50.872402
M00934_Edges4,0.000000,18.000000,56.438895,0.000000,18.000000,50.872403
M03528_Edges4,0.000000,17.999997,56.438893,0.000000,18.000000,50.872403
M10302_Edges4,0.000000,18.000000,56.438896,0.000000,18.000000,50.872403
M25888_Edges4,0.000000,18.000000,56.438896,0.000000,18.000000,50.872403
M25373_Edges5,0.024796,19.974960,62.684844,0.058305,19.931037,56.455929
M25243_Edges5,0.024796,19.974960,62.684844,0.058305,19.931037,56.455929


Percebe-se que o Top 7 tem uma discrepância em BIC relevante a partir do 8º modelo

In [10]:
top_7 = modelos_filtro2.sort_values("BIC_autismo").head(7)
top_7_nomes = top_7.index.values
top_7 

,RMSEA_controle,AIC_controle,BIC_controle,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,
M08682_Edges4,0.071795,17.915579,56.354475,0.067727,17.903044,50.775447
M08812_Edges4,0.071795,17.915579,56.354475,0.067727,17.903045,50.775447
M00001_Edges4,0.000000,18.000000,56.438896,0.000000,17.999999,50.872402
M00934_Edges4,0.000000,18.000000,56.438895,0.000000,18.000000,50.872403
M03528_Edges4,0.000000,17.999997,56.438893,0.000000,18.000000,50.872403
M10302_Edges4,0.000000,18.000000,56.438896,0.000000,18.000000,50.872403
M25888_Edges4,0.000000,18.000000,56.438896,0.000000,18.000000,50.872403


### 4) Verificação de arestas estatisticamente significativas

In [11]:
def roda_os_modelos(df, modelos, ajuste_p=None, retornar_inspection=False):    
    df_resultados = pd.DataFrame()
    dicionario = dict()

    for nome, modelo in modelos.items():
        try:
            formula = modelo["formula"]
        except:
            formula = modelo
        
        model = semopy.Model(formula)
        model.fit(df, obj="MLW")
        stats = semopy.stats.calc_stats(model)
        df_resultados = pd.concat([df_resultados, stats])
        if retornar_inspection:
            edges_inspection = model.inspect()
            edges_inspection = edges_inspection[edges_inspection["op"] == "~"]
            if ajuste_p:
                p_valores = edges_inspection["p-value"]
                rejeitados, p_corrigidos_fdr, _, _ = multipletests(p_valores, alpha=0.05, method=ajuste_p)
                edges_inspection["p-value-adj"] = p_corrigidos_fdr
                edges_inspection["tipo_correcao"] = ajuste_p
                edges_inspection["rejeita"] = rejeitados
            dicionario[nome] = edges_inspection
            
   
    df_resultados["Modelo"] = modelos.keys()
    df_resultados.set_index("Modelo", inplace=True)

    if retornar_inspection:
        return df_resultados, dicionario

    return df_resultados

In [12]:
lista_modelos_interesse = top_7_nomes

modelos_interesse = {m:v for m,v in formula_modelos_conexos.items() if m in lista_modelos_interesse}

#### 4.1) Arestas indicadas pelo Conjunto de Controle

In [13]:
_, dic_arestas_controle = roda_os_modelos(df_controle_prep, 
                                        modelos_interesse,
                                        ajuste_p="bonferroni", 
                                        retornar_inspection=True)
for m in modelos_interesse.keys():
    print(m)
    display(dic_arestas_controle[m])

M00001_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Control,~,Visual,0.101172,0.040732,2.483864,1.299654e-02,5.198616e-02,bonferroni,False
1,Control,~,Somatomotor,0.212962,0.041278,5.159241,2.479526e-07,9.918103e-07,bonferroni,True
2,Control,~,Default_mode,0.333623,0.039591,8.426640,0.000000e+00,0.000000e+00,bonferroni,True
3,Control,~,Cerebellar,0.156906,0.039617,3.960596,7.476302e-05,2.990521e-04,bonferroni,True


M00934_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Cerebellar,~,Visual,0.261108,0.042843,6.094500,1.097802e-09,4.391207e-09,bonferroni,True
1,Cerebellar,~,Somatomotor,0.232740,0.044625,5.215502,1.833205e-07,7.332819e-07,bonferroni,True
2,Cerebellar,~,Default_mode,-0.080830,0.045473,-1.777530,7.548114e-02,3.019245e-01,bonferroni,False
3,Cerebellar,~,Control,0.183324,0.046351,3.955085,7.650735e-05,3.060294e-04,bonferroni,True


M03528_Edges4


...

,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Default_mode,~,Somatomotor,0.227999,0.041215,5.531958,3.166764e-08,1.266706e-07,bonferroni,True
1,Default_mode,~,Control,0.384524,0.041215,9.329722,0.000000e+00,0.000000e+00,bonferroni,True
2,Visual,~,Default_mode,0.297268,0.039025,7.617375,2.597922e-14,1.039169e-13,bonferroni,True
3,Visual,~,Cerebellar,0.326281,0.039024,8.361048,0.000000e+00,0.000000e+00,bonferroni,True


M10302_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Somatomotor,~,Visual,0.137922,0.041685,3.308704,9.372887e-04,3.749155e-03,bonferroni,True
1,Somatomotor,~,Default_mode,0.198418,0.042473,4.671599,2.988633e-06,1.195453e-05,bonferroni,True
2,Somatomotor,~,Cerebellar,0.210254,0.040297,5.217574,1.812823e-07,7.251292e-07,bonferroni,True
3,Somatomotor,~,Control,0.224701,0.043611,5.152348,2.572444e-07,1.028978e-06,bonferroni,True


M25888_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Visual,~,Somatomotor,0.147016,0.044437,3.308391,9.383359e-04,3.753343e-03,bonferroni,True
1,Visual,~,Default_mode,0.197659,0.043912,4.501230,6.756149e-06,2.702460e-05,bonferroni,True
2,Visual,~,Cerebellar,0.250934,0.041244,6.084193,1.170796e-09,4.683184e-09,bonferroni,True
3,Visual,~,Control,0.114442,0.045876,2.494590,1.261028e-02,5.044112e-02,bonferroni,False


#### 4.2) Arestas indicadas pelo Conjunto de Autismo

In [14]:
_, dic_arestas_autismo = roda_os_modelos(df_autismo_prep, 
                                        modelos_interesse,
                                        ajuste_p="bonferroni", 
                                        retornar_inspection=True)
for m in modelos_interesse.keys():
    print(m)
    display(dic_arestas_autismo[m])

M00001_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Control,~,Visual,0.115832,0.050322,2.301828,2.134488e-02,8.537952e-02,bonferroni,False
1,Control,~,Somatomotor,0.210378,0.055829,3.768237,1.644047e-04,6.576186e-04,bonferroni,True
2,Control,~,Default_mode,0.420271,0.053312,7.883272,3.108624e-15,1.243450e-14,bonferroni,True
3,Control,~,Cerebellar,0.168629,0.048525,3.475114,5.106366e-04,2.042546e-03,bonferroni,True


M00934_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Cerebellar,~,Visual,0.211197,0.059417,3.554457,3.787608e-04,1.515043e-03,bonferroni,True
1,Cerebellar,~,Somatomotor,0.383588,0.064499,5.947149,2.728532e-09,1.091413e-08,bonferroni,True
2,Cerebellar,~,Default_mode,-0.204911,0.069284,-2.957561,3.100831e-03,1.240332e-02,bonferroni,True
3,Cerebellar,~,Control,0.241052,0.069372,3.474790,5.112535e-04,2.045014e-03,bonferroni,True


M03528_Edges4


...

,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Default_mode,~,Somatomotor,0.299931,0.053034,5.655477,1.554140e-08,6.216559e-08,bonferroni,True
1,Default_mode,~,Control,0.471764,0.053034,8.895563,0.000000e+00,0.000000e+00,bonferroni,True
2,Visual,~,Default_mode,0.437040,0.051610,8.468145,0.000000e+00,0.000000e+00,bonferroni,True
3,Visual,~,Cerebellar,0.280432,0.051610,5.433730,5.518805e-08,2.207522e-07,bonferroni,True


M10302_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Somatomotor,~,Visual,0.074998,0.052409,1.431023,1.524236e-01,6.096943e-01,bonferroni,False
1,Somatomotor,~,Default_mode,0.313006,0.058039,5.392997,6.929194e-08,2.771677e-07,bonferroni,True
2,Somatomotor,~,Cerebellar,0.287730,0.048392,5.945848,2.750294e-09,1.100117e-08,bonferroni,True
3,Somatomotor,~,Control,0.225384,0.059882,3.763784,1.673617e-04,6.694468e-04,bonferroni,True


M25888_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Visual,~,Somatomotor,0.095195,0.066470,1.432152,0.152100,0.608401,bonferroni,False
1,Visual,~,Default_mode,0.301844,0.066246,4.556377,0.000005,0.000021,bonferroni,True
2,Visual,~,Cerebellar,0.200937,0.056539,3.553946,0.000379,0.001518,bonferroni,True
3,Visual,~,Control,0.158055,0.068456,2.308853,0.020952,0.083807,bonferroni,False


### 5) Poda de modelos e comparação com modelos não podados

#### 5.1) Poda de arestas

In [15]:
modelos_podados_bonf = {
    "M00001_Edges4_podado": "Visual ~~ Visual\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~ 0*Visual + Somatomotor + Default_mode + Cerebellar",
    "M00934_Edges4_podado": "Visual ~~ Visual\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nControl ~~ Control\nCerebellar ~ Visual + Somatomotor + 0*Default_mode + Control",
    "M03528_Edges4_podado": "Visual ~~ Visual\nSomatomotor ~~ Somatomotor\nDefault_mode ~ Visual + Somatomotor + 0*Cerebellar + Control\nCerebellar ~~ Cerebellar\nControl ~~ Control",
    "M10302_Edges4_podado": "Visual ~~ Visual\nSomatomotor ~ 0*Visual + Default_mode + Cerebellar + Control\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control",
    "M25888_Edges4_podado": "Visual ~ Somatomotor + Default_mode + Cerebellar + 0*Control\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control",
    "M25888_Edges4_podado_2": "Visual ~ 0*Somatomotor + Default_mode + Cerebellar + 0*Control\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control"
}

modelos_podados = {
    "M00934_Edges4_podado": "Visual ~~ Visual\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nControl ~~ Control\nCerebellar ~ Visual + Somatomotor + 0*Default_mode + Control",
    "M03528_Edges4_podado": "Visual ~~ Visual\nSomatomotor ~~ Somatomotor\nDefault_mode ~ Visual + Somatomotor + 0*Cerebellar + Control\nCerebellar ~~ Cerebellar\nControl ~~ Control",
    "M10302_Edges4_podado": "Visual ~~ Visual\nSomatomotor ~ 0*Visual + Default_mode + Cerebellar + Control\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control",
    "M25888_Edges4_podado": "Visual ~ 0*Somatomotor + Default_mode + Cerebellar + Control\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control",
}

In [16]:
res_autismo, dic_arestas_autismo = roda_os_modelos(
                                        df_autismo_prep, 
                                        modelos_podados,
                                        retornar_inspection=True)
res_autismo = res_autismo.reset_index()[colunas_interesse].set_index("Modelo")


res_controle, dic_arestas_controle = roda_os_modelos(
                                        df_controle_prep, 
                                        modelos_podados,
                                        retornar_inspection=True)
res_controle = res_controle.reset_index()[colunas_interesse].set_index("Modelo")



res_podados = pd.merge(res_controle, 
         res_autismo, 
         how="inner", 
         left_index=True, 
         right_index=True, 
         suffixes=["_controle", "_autismo"])


display(res_podados)

for m in modelos_podados.keys():
    print(f"\n{m}:")
    print("CONTROLE")
    display(dic_arestas_controle[m])

    print("\nAUTISMO")
    display(dic_arestas_autismo[m])

,RMSEA_controle,AIC_controle,BIC_controle,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,
M00934_Edges4_podado,0.000000,15.988139,50.156046,0.028491,15.939553,45.159467
M03528_Edges4_podado,0.000000,15.988139,50.156046,0.028491,15.939553,45.159467
M10302_Edges4_podado,0.032201,15.959046,50.126953,0.000000,15.985659,45.205572
M25888_Edges4_podado,0.032201,15.959046,50.126953,0.000000,15.985659,45.205572



M00934_Edges4_podado:
CONTROLE


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Cerebellar,~,Visual,0.246933,0.042269,5.841895,0.0
1,Cerebellar,~,Somatomotor,0.217937,0.043959,4.957703,0.000001
2,Cerebellar,~,Default_mode,0.000000,-,-,-
3,Cerebellar,~,Control,0.155841,0.043768,3.560657,0.00037



AUTISMO


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Cerebellar,~,Visual,0.170136,0.058662,2.900285,0.003728
1,Cerebellar,~,Somatomotor,0.332789,0.063132,5.271331,0.0
2,Cerebellar,~,Default_mode,0.000000,-,-,-
3,Cerebellar,~,Control,0.158600,0.06448,2.459675,0.013906


...

,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Somatomotor,~,Visual,0.000000,-,-,-
1,Somatomotor,~,Default_mode,0.338159,0.055529,6.089789,0.0
2,Somatomotor,~,Cerebellar,0.304860,0.04703,6.482203,0.0
3,Somatomotor,~,Control,0.239095,0.059334,4.029638,0.000056



M25888_Edges4_podado:
CONTROLE


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Visual,~,Somatomotor,0.000000,-,-,-
1,Visual,~,Default_mode,0.231637,0.043143,5.368992,0.0
2,Visual,~,Cerebellar,0.287832,0.04013,7.172468,0.0
3,Visual,~,Control,0.150542,0.045019,3.343946,0.000826



AUTISMO


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Visual,~,Somatomotor,0.000000,-,-,-
1,Visual,~,Default_mode,0.334177,0.062547,5.342852,0.0
2,Visual,~,Cerebellar,0.229869,0.052974,4.339286,0.000014
3,Visual,~,Control,0.180521,0.066833,2.701085,0.006911


### 6) Teste de diferença de parâmetros das arestas:

#### 6.1) Teste z

In [17]:
modelos_podados.update({m:v["formula"] for m, v in modelos_interesse.items()})

In [18]:
for nome, formula in modelos_podados.items():

    # --- Grupo Controle ---
    model_controle = semopy.Model(formula)
    model_controle.fit(df_controle_prep)
    arestas_controle = model_controle.inspect()

    # Filtro e conversões (usando .copy() para evitar warnings do Pandas)
    arestas_controle = arestas_controle[(arestas_controle["op"] == "~") & (arestas_controle["Std. Err"] != "-")].copy()
    arestas_controle["Estimate"] = arestas_controle["Estimate"].astype(float)
    arestas_controle["Std. Err"] = arestas_controle["Std. Err"].astype(float)

    # --- Grupo Autismo ---
    model_autismo = semopy.Model(formula)
    model_autismo.fit(df_autismo_prep)
    arestas_autismo = model_autismo.inspect()
    
    # Filtro e conversões
    arestas_autismo = arestas_autismo[(arestas_autismo["op"] == "~") & (arestas_autismo["Std. Err"] != "-")].copy()
    arestas_autismo["Estimate"] = arestas_autismo["Estimate"].astype(float)
    arestas_autismo["Std. Err"] = arestas_autismo["Std. Err"].astype(float)

    # --- Comparação Segura (Merge) ---
    # O merge garante que a aresta X->Y do controle só será subtraída da aresta X->Y do autismo
    df_comparacao = pd.merge(
        arestas_controle, 
        arestas_autismo, 
        on=["lval", "op", "rval"], 
        suffixes=('_ctrl', '_tea')
    )

    # Cálculo do Z e P-valor usando as colunas mescladas
    z_diff = (df_comparacao["Estimate_ctrl"] - df_comparacao["Estimate_tea"]) / np.sqrt((df_comparacao["Std. Err_ctrl"]**2) + (df_comparacao["Std. Err_tea"]**2))
    p_valor = 2 * (1 - norm.cdf(np.abs(z_diff)))
    
    # Guardando os resultados no dataframe
    df_comparacao["z_diff"] = z_diff
    df_comparacao["p_valor"] = p_valor
    
    print(nome)
    display(df_comparacao)

M00934_Edges4_podado


,lval,op,rval,Estimate_ctrl,Std. Err_ctrl,z-value_ctrl,p-value_ctrl,Estimate_tea,Std. Err_tea,z-value_tea,p-value_tea,z_diff,p_valor
0,Cerebellar,~,Visual,0.246933,0.042269,5.841895,0.0,0.170136,0.058662,2.900285,0.003728,1.062127,0.288178
1,Cerebellar,~,Somatomotor,0.217937,0.043959,4.957703,0.000001,0.332789,0.063132,5.271331,0.0,-1.492970,0.135445
2,Cerebellar,~,Control,0.155841,0.043768,3.560657,0.00037,0.158600,0.064480,2.459675,0.013906,-0.035404,0.971758


M03528_Edges4_podado


,lval,op,rval,Estimate_ctrl,Std. Err_ctrl,z-value_ctrl,p-value_ctrl,Estimate_tea,Std. Err_tea,z-value_tea,p-value_tea,z_diff,p_valor
0,Default_mode,~,Visual,0.168610,0.040301,4.183804,0.000029,0.200356,0.049395,4.056216,0.00005,-0.497973,0.618503
1,Default_mode,~,Somatomotor,0.183836,0.041912,4.386243,0.000012,0.247415,0.053159,4.654283,0.000003,-0.939218,0.347619
2,Default_mode,~,Control,0.343209,0.041729,8.224666,0.0,0.402979,0.054294,7.422185,0.0,-0.872850,0.382745


M10302_Edges4_podado


...

,lval,op,rval,Estimate_ctrl,Std. Err_ctrl,z-value_ctrl,p-value_ctrl,Estimate_tea,Std. Err_tea,z-value_tea,p-value_tea,z_diff,p_valor
0,Default_mode,~,Somatomotor,0.227999,0.041215,5.531958,3.166764e-08,0.299931,0.053034,5.655477,1.554140e-08,-1.070954,0.284190
1,Default_mode,~,Control,0.384524,0.041215,9.329722,0.000000e+00,0.471764,0.053034,8.895563,0.000000e+00,-1.298883,0.193984
2,Visual,~,Default_mode,0.297268,0.039025,7.617375,2.597922e-14,0.437040,0.051610,8.468145,0.000000e+00,-2.160198,0.030757
3,Visual,~,Cerebellar,0.326281,0.039024,8.361048,0.000000e+00,0.280432,0.051610,5.433730,5.518805e-08,0.708618,0.478562


M10302_Edges4


,lval,op,rval,Estimate_ctrl,Std. Err_ctrl,z-value_ctrl,p-value_ctrl,Estimate_tea,Std. Err_tea,z-value_tea,p-value_tea,z_diff,p_valor
0,Somatomotor,~,Visual,0.137922,0.041685,3.308704,9.372887e-04,0.074998,0.052409,1.431023,1.524236e-01,0.939655,0.347395
1,Somatomotor,~,Default_mode,0.198418,0.042473,4.671599,2.988633e-06,0.313006,0.058039,5.392997,6.929194e-08,-1.593266,0.111100
2,Somatomotor,~,Cerebellar,0.210254,0.040297,5.217574,1.812823e-07,0.287730,0.048392,5.945848,2.750294e-09,-1.230305,0.218583
3,Somatomotor,~,Control,0.224701,0.043611,5.152348,2.572444e-07,0.225384,0.059882,3.763784,1.673617e-04,-0.009215,0.992647


M25888_Edges4


,lval,op,rval,Estimate_ctrl,Std. Err_ctrl,z-value_ctrl,p-value_ctrl,Estimate_tea,Std. Err_tea,z-value_tea,p-value_tea,z_diff,p_valor
0,Visual,~,Somatomotor,0.147016,0.044437,3.308391,9.383359e-04,0.095195,0.066470,1.432152,0.152100,0.648130,0.516901
1,Visual,~,Default_mode,0.197659,0.043912,4.501230,6.756149e-06,0.301844,0.066246,4.556377,0.000005,-1.310853,0.189907
2,Visual,~,Cerebellar,0.250934,0.041244,6.084193,1.170796e-09,0.200937,0.056539,3.553946,0.000379,0.714413,0.474972
3,Visual,~,Control,0.114442,0.045876,2.494590,1.261028e-02,0.158055,0.068456,2.308853,0.020952,-0.529234,0.596643
